In [ ]:
from databricks.connect import DatabricksSession
spark = DatabricksSession.builder.getOrCreate()
from yellow_taxi.yellow_taxi_utils import  enrich_yellow_trips, merge_insert
from helper_utils.functions import get_incremental_data
from delta.tables import DeltaTable

In [ ]:
# variables
# interactive widgets for catalog and schema names

dbutils.widgets.text("catalog", "nyctaxi_project_dev")
dbutils.widgets.text("silver_schema", "silver")

# dynamic variables for catalog and schema names
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("silver_schema")

silver_trips_tbl = f"{catalog}.{schema}.yellow_trips_normalized"
silver_lookup_tbl = f"{catalog}.{schema}.yellow_taxi_silver_lookup"
payment_type_tbl = f"{catalog}.{schema}.payment_type"
vendor_tbl = f"{catalog}.{schema}.vendor"
rates_tbl = f"{catalog}.{schema}.rates"
target_tbl = f"{catalog}.{schema}.yellow_trips_enriched"

In [3]:
# load tables to df
silver_trips = spark.table(silver_trips_tbl)
silver_lookup = spark.table(silver_lookup_tbl)
payment_type = spark.table(payment_type_tbl)
vendor = spark.table(vendor_tbl)
rates = spark.table(rates_tbl)

In [4]:
# load only incremental trips since the last run
incremental = get_incremental_data(spark, target_tbl, silver_trips)

In [5]:
# enrich yellow trips with the lookup data

enriched = enrich_yellow_trips(incremental, vendor, payment_type, rates, silver_lookup)


In [6]:
# create table if not exists and merge changes 
if not spark.catalog.tableExists(target_tbl):
    enriched.write.mode("overwrite").saveAsTable(target_tbl)
else:
    dt = DeltaTable.forName(spark, target_tbl)
    merge_insert(enriched, dt)
